In [6]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error


for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/russian-car-plates-prices-prediction/sample_submission.csv
/kaggle/input/competitions/russian-car-plates-prices-prediction/supplemental_english.py
/kaggle/input/competitions/russian-car-plates-prices-prediction/supplemental_russian.py
/kaggle/input/competitions/russian-car-plates-prices-prediction/train.csv
/kaggle/input/competitions/russian-car-plates-prices-prediction/test.csv


In [7]:
import os

for folder, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(folder, filename))

/kaggle/input/competitions/russian-car-plates-prices-prediction/sample_submission.csv
/kaggle/input/competitions/russian-car-plates-prices-prediction/supplemental_english.py
/kaggle/input/competitions/russian-car-plates-prices-prediction/supplemental_russian.py
/kaggle/input/competitions/russian-car-plates-prices-prediction/train.csv
/kaggle/input/competitions/russian-car-plates-prices-prediction/test.csv


In [8]:
data_dir = '/kaggle/input/competitions/russian-car-plates-prices-prediction'

train = pd.read_csv(f'{data_dir}/train.csv')
test = pd.read_csv(f'{data_dir}/test.csv')
sample_submission = pd.read_csv(f'{data_dir}/sample_submission.csv')

print("Train", train.shape)
print('Test', test.shape)
print("Sample Submission", sample_submission.shape)

display(train.head())


missing = train.isna().sum()
missing = missing[missing>0].sort_values(ascending=False)
display(missing)
print('Training duplicate rows:', train.duplicated().sum())

Train (51635, 4)
Test (7695, 4)
Sample Submission (7695, 2)


,id,plate,date,price
0,1,X059CP797,2024-12-26 00:00:00,65000
1,2,Y800MH790,2024-07-12 21:31:37,100000
2,3,A212TX77,2024-04-18 00:00:00,290000
3,4,P001AY199,2025-01-03 00:27:15,680000
4,5,P001AY199,2025-01-10 09:32:41,750000


Series([], dtype: int64)

Training duplicate rows: 0


In [9]:
# Convert date to datetime values
train['date'] = pd.to_datetime(train['date'], errors = 'coerce')
test['date'] = pd.to_datetime(test['date'], errors = 'coerce')

print('Training date range:', train['date'].min(), 'to', train['date'].max())
print('Test date range:', test['date'].min(), 'to', test['date'].max())
print('Unparsed training dates:', train['date'].isna().sum())
print('Unparsed testing dates:', test['date'].isna().sum())

print('\nTraining prices:')
display(train['price'].describe(percentiles=[0.50,0.75,0.90,0.95,0.99]))

print('Price Skew:', round(train['price'].skew(), 2))
print('Unique plate in training:', train['plate'].nunique())

print('\nMost frequently listed plates:')
display(train['plate'].value_counts().head(10))

Training date range: 2021-02-17 21:21:56 to 2025-02-12 00:00:00
Test date range: 2021-02-17 21:22:02 to 2025-02-12 00:00:00
Unparsed training dates: 0
Unparsed testing dates: 0

Training prices:


count    5.163500e+04
mean     4.532253e+05
std      1.793287e+06
min      2.000000e+04
50%      1.500000e+05
75%      3.600000e+05
90%      8.500000e+05
95%      1.500000e+06
99%      5.000000e+06
max      1.000000e+08
Name: price, dtype: float64

Price Skew: 26.39
Unique plate in training: 43605

Most frequently listed plates:


plate
A949MP190    119
X582XX190     82
B777OC90      72
P606PP190     55
E750OX750     51
O595OO150     45
E555EE90      43
B333XP750     40
H001BB150     36
T777OB90      36
Name: count, dtype: int64

In [10]:
print('Test Columns:', test.columns.tolist())
display(test.head())
print('\nMissing Values in test:')
display(test.isna().sum())
print('\nSample Submission columns:', sample_submission.columns.tolist())
display(sample_submission.head())

Test Columns: ['id', 'plate', 'date', 'price']


,id,plate,date,price
0,51636,P700TT790,2025-01-27,NaN
1,51637,M081TX797,2025-02-10,NaN
2,51638,T333HX777,2025-02-11,NaN
3,51639,H744BH977,2025-02-03,NaN
4,51640,X066EM777,2025-02-12,NaN



Missing Values in test:


id          0
plate       0
date        0
price    7695
dtype: int64


Sample Submission columns: ['id', 'price']


,id,price
0,51636,100000
1,51637,100000
2,51638,100000
3,51639,100000
4,51640,100000


In [11]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor

def make_plate_features(data):
    plate = data["plate"].fillna("").astype(str).str.upper().str.strip()
    date = pd.to_datetime(data["date"], errors="coerce")

    features = pd.DataFrame(index=data.index)

    # Plate characteristics
    digits = plate.str[1:4]
    letters = plate.str[4:6]
    region = plate.str.extract(r"(\d{2,3})$")[0]

    features["plate"] = plate
    features["first_letter"] = plate.str[0]
    features["plate_digits"] = digits
    features["middle_letters"] = letters
    features["region_code"] = region.fillna("Unknown")

    features["repeated_digit_count"] = digits.map(
        lambda value: 3 - len(set(value)) if len(value) == 3 else 0
    )
    features["all_digits_same"] = digits.map(
        lambda value: int(len(value) == 3 and len(set(value)) == 1)
    )
    features["digits_palindrome"] = digits.map(
        lambda value: int(len(value) == 3 and value == value[::-1])
    )
    features["letters_match"] = (
        (letters.str[0] == letters.str[1]).fillna(False).astype(int)
    )

    # Date characteristics
    features["year"] = date.dt.year
    features["month"] = date.dt.month
    features["day_of_week"] = date.dt.dayofweek
    features["day"] = date.dt.day
    features["hour"] = date.dt.hour
    features["days_since_start"] = (
        date - pd.Timestamp("2021-02-17")
    ).dt.days

    # CatBoost needs categorical values represented as strings
    categorical_features = [
        "plate",
        "first_letter",
        "plate_digits",
        "middle_letters",
        "region_code",
    ]
    for column in categorical_features:
        features[column] = features[column].fillna("Unknown").astype(str)

    return features, categorical_features


X, categorical_features = make_plate_features(train)
y = train["price"]
X_test_model, _ = make_plate_features(test)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = CatBoostRegressor(
    iterations=1500,
    depth=7,
    learning_rate=0.05,
    loss_function="MAE",
    eval_metric="SMAPE",
    random_seed=42,
    verbose=200,
    allow_writing_files=False,
)

model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=150,
    verbose=200,
)

valid_predictions = np.maximum(model.predict(X_valid), 0)

smape = 100 * np.mean(
    2 * np.abs(y_valid.to_numpy() - valid_predictions)
    / (np.abs(y_valid.to_numpy()) + np.abs(valid_predictions) + 1e-8)
)

print("Validation SMAPE:", round(smape, 3))
print("Best iteration:", model.get_best_iteration())

0:	learn: 75.4089996	test: 75.8667650	best: 75.8667650 (0)	total: 121ms	remaining: 3m
200:	learn: 40.0130974	test: 40.2278765	best: 40.2278765 (200)	total: 8.54s	remaining: 55.2s
400:	learn: 38.0093454	test: 38.9819460	best: 38.9819460 (400)	total: 17.4s	remaining: 47.8s
600:	learn: 36.8672773	test: 38.5336044	best: 38.5336044 (600)	total: 26.2s	remaining: 39.3s
800:	learn: 36.0688120	test: 38.3248059	best: 38.3202544 (785)	total: 35.2s	remaining: 30.7s
1000:	learn: 35.4399183	test: 38.1725536	best: 38.1718730 (997)	total: 44.4s	remaining: 22.1s
1200:	learn: 34.8187951	test: 38.0551880	best: 38.0551880 (1200)	total: 53.8s	remaining: 13.4s
1400:	learn: 34.3181593	test: 37.9585520	best: 37.9580751 (1394)	total: 1m 3s	remaining: 4.47s
1499:	learn: 34.1082414	test: 37.9223602	best: 37.9219739 (1498)	total: 1m 7s	remaining: 0us

bestTest = 37.92197391
bestIteration = 1498

Shrink model to first 1499 iterations.
Validation SMAPE: 37.922
Best iteration: 1498


In [12]:
final_iterations = model.get_best_iteration() + 1

final_model = CatBoostRegressor(
    iterations=final_iterations,
    depth=7,
    learning_rate=0.05,
    loss_function="MAE",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

final_model.fit(
    X,
    y,
    cat_features=categorical_features,
)

test_predictions = np.maximum(
    final_model.predict(X_test_model),
    0
)

submission = sample_submission.copy()
submission["price"] = test_predictions

submission.to_csv(
    "/kaggle/working/submission_plate_catboost.csv",
    index=False
)

print("Submission shape:", submission.shape)
print(submission.head())
print("Saved: /kaggle/working/submission_plate_catboost.csv")

Submission shape: (7695, 2)
      id          price
0  51636  216202.129674
1  51637   56055.982058
2  51638  601423.141754
3  51639   63047.103124
4  51640  180900.223317
Saved: /kaggle/working/submission_plate_catboost.csv


In [13]:
from IPython.display import FileLink, display

display(FileLink("submission_plate_catboost.csv"))

/kaggle/working/submission_plate_catboost.csv